In [3]:
!pip install tcia-utils
import os
from tcia_utils import nbia

os.makedirs("NSCLC_External_Dataset", exist_ok=True)

print("Fetching list of patients from TCIA...")
series_data = nbia.getSeries(collection="NSCLC-Radiomics")

print("Starting Massive 35GB Download... (Progress bar will appear below)")
# Notice we are passing series_data here!
nbia.downloadSeries(series_data, path="NSCLC_External_Dataset")
print("Download Complete!")

2026-10-01 05:13:31,132:INFO:Calling getSeries with parameters {'Collection': 'NSCLC-Radiomics'}


Fetching list of patients from TCIA...


2026-10-01 05:13:36,075:INFO:Directory 'NSCLC_External_Dataset' already exists.
2026-10-01 05:13:36,078:INFO:Found 0 previously downloaded series.
2026-10-01 05:13:36,080:INFO:Attempting to download 1265 new series.
2026-10-01 05:13:36,084:INFO:Downloading... https://nbia.cancerimagingarchive.net/nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6.1.4.1.32722.99.99.320898527671900265039224224949289088459
2026-10-01 05:13:36,088:INFO:Downloading... https://nbia.cancerimagingarchive.net/nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6.1.4.1.32722.99.99.71621653125201582124240564508842688465
2026-10-01 05:13:36,097:INFO:Downloading... https://nbia.cancerimagingarchive.net/nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6.1.4.1.32722.99.99.138522260934437218114778023563031054616
2026-10-01 05:13:36,105:INFO:Downloading... https://nbia.cancerimagingarchive.net/nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6

Starting Massive 35GB Download... (Progress bar will appear below)


2026-10-01 05:13:54,713:INFO:Downloading... https://nbia.cancerimagingarchive.net/nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6.1.4.1.32722.99.99.25104106093039357825811136704867874087
2026-10-01 05:14:01,560:INFO:Downloading... https://nbia.cancerimagingarchive.net/nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6.1.4.1.32722.99.99.329732333018868310332225541275420754643
2026-10-01 05:14:03,022:INFO:Downloading... https://nbia.cancerimagingarchive.net/nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6.1.4.1.32722.99.99.175539867238710307908179945225731049379
2026-10-01 05:14:03,081:INFO:Downloading... https://nbia.cancerimagingarchive.net/nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6.1.4.1.32722.99.99.40812172850593977264671028634455343097
2026-10-01 05:14:06,624:INFO:Downloading... https://nbia.cancerimagingarchive.net/nbia-api/services/v4/getImage?NewFileNames=Yes&SeriesInstanceUID=1.3.6.1.4.1

Download Complete!


In [7]:
!pip install rt-utils scikit-image pydicom scipy --user --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 55.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 81.7 MB/s  0:00:00m0:00:0100:01m
   ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [tifffile]  WARNING: The scripts lsm2bin, tiff2fsspec, tiffcomment and tifffile are installed in '/root/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/7 [pydicom]  WARNING: The script pydicom is installed in '/root/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━ 4/7 [imageio]  WARNING: The scripts imageio_download_bin and imageio_remove_bin are installed in '/root/.local/bin' which is not on PATH.
  Con

In [ ]:
import os
os._exit(00)

In [4]:
import pydicom as dicom
import numpy as np
import os
import glob
from skimage import measure
from skimage.draw import polygon
import scipy.ndimage
import csv
from multiprocessing import Pool, cpu_count

def get_mask_from_rtstruct(rt_file_path, dicom_slices, target_roi_keyword='GTV'):
    rtstruct = dicom.dcmread(rt_file_path)
    image_shape = dicom_slices[0].pixel_array.shape
    num_slices = len(dicom_slices)
    mask = np.zeros((image_shape[0], image_shape[1], num_slices), dtype=np.uint8)
    slice_positions = {float(s.ImagePositionPatient[2]): i for i, s in enumerate(dicom_slices)}
    roi_number = None
    for roi in rtstruct.StructureSetROISequence:
        if target_roi_keyword.upper() in roi.ROIName.upper():
            roi_number = roi.ROINumber
            break
    if roi_number is None:
        return mask
    for roi_contour in rtstruct.ROIContourSequence:
        if roi_contour.ReferencedROINumber == roi_number:
            if not hasattr(roi_contour, 'ContourSequence'):
                break
            for contour in roi_contour.ContourSequence:
                pts = np.array(contour.ContourData).reshape(-1, 3)
                z = float(pts[0, 2])
                closest_z = min(slice_positions.keys(), key=lambda x: abs(x - z))
                if abs(closest_z - z) > 2.0:
                    continue
                slice_idx = slice_positions[closest_z]
                row_spacing = float(dicom_slices[slice_idx].PixelSpacing[0])
                col_spacing = float(dicom_slices[slice_idx].PixelSpacing[1])
                origin = dicom_slices[slice_idx].ImagePositionPatient
                col_coords = ((pts[:, 0] - float(origin[0])) / col_spacing).astype(int)
                row_coords = ((pts[:, 1] - float(origin[1])) / row_spacing).astype(int)
                rr, cc = polygon(row_coords, col_coords, shape=(image_shape[0], image_shape[1]))
                mask[rr, cc, slice_idx] = 1
            break
    return mask

def load_scan(path):
    slices = [dicom.dcmread(s) for s in glob.glob(path+'/*.dcm')]
    slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
    try:
        slice_thickness = np.abs(slices[0].ImagePositionPatient[2] - slices[1].ImagePositionPatient[2])
    except:
        slice_thickness = np.abs(slices[0].SliceLocation - slices[1].SliceLocation)
    for s in slices:
        s.SliceThickness = slice_thickness
    return slices

def center_crop_64(vol):
    cx, cy, cz = np.array(vol.shape) // 2
    x0 = max(0, cx-32); x1 = x0+64
    y0 = max(0, cy-32); y1 = y0+64
    z0 = max(0, cz-32); z1 = z0+64
    if x1 > vol.shape[0]: x0, x1 = vol.shape[0]-64, vol.shape[0]
    if y1 > vol.shape[1]: y0, y1 = vol.shape[1]-64, vol.shape[1]
    if z1 > vol.shape[2]: z0, z1 = vol.shape[2]-64, vol.shape[2]
    return vol[x0:x1, y0:y1, z0:z1]

def process_patient(args):
    pid, ct_dir, rt_file, save_path = args
    try:
        dicom_slices = load_scan(ct_dir)
        spacing = np.array([dicom_slices[0].PixelSpacing[0],
                            dicom_slices[0].PixelSpacing[1],
                            dicom_slices[0].SliceThickness]).astype(np.float32)
        resize_factor = tuple((spacing / 1.0).tolist())

        image_cpu = np.array([s.pixel_array.astype(np.int32) * int(s.RescaleSlope) + int(s.RescaleIntercept)
                              for s in dicom_slices]).transpose(1, 2, 0).astype(np.float32)

        mask_cpu = get_mask_from_rtstruct(rt_file, dicom_slices)
        L_props = measure.regionprops(measure.label(mask_cpu))
        center_pos = [np.array(p.centroid).astype(int) for p in L_props]

        if not center_pos:
            return pid, 0, []

        ppath = os.path.join(save_path, pid)
        os.makedirs(ppath, exist_ok=True)

        rows = []
        Nodule_num = 0
        for pos in center_pos:
            Nodule_num += 1
            patch = 80
            r0, r1 = max(0, pos[0]-patch), min(image_cpu.shape[0], pos[0]+patch)
            c0, c1 = max(0, pos[1]-patch), min(image_cpu.shape[1], pos[1]+patch)
            z0, z1 = max(0, pos[2]-patch), min(image_cpu.shape[2], pos[2]+patch)

            img_patch  = scipy.ndimage.zoom(image_cpu[r0:r1, c0:c1, z0:z1], resize_factor, order=1)
            mask_patch = scipy.ndimage.zoom(mask_cpu[r0:r1, c0:c1, z0:z1].astype(np.float32), resize_factor, order=1)

            img_patch = center_crop_64(img_patch)
            mask_patch = center_crop_64(mask_patch)

            # Normalize HU
            img_patch = np.clip((img_patch - (-1200.0)) / (600.0 - (-1200.0)), 0, 1) * 255.0
            img_patch = (img_patch - 128.0) / 128.0

            roi_path  = os.path.join(ppath, f"{pid}_{Nodule_num}_roi.npy")
            mask_path = os.path.join(ppath, f"{pid}_{Nodule_num}_label.npy")
            np.save(roi_path,  img_patch.astype(np.float32))
            np.save(mask_path, mask_patch.astype(np.float32))
            rows.append([pid, Nodule_num, roi_path, mask_path, None])

        return pid, Nodule_num, rows

    except Exception as e:
        return pid, -1, str(e)


# --- MAIN ---
home_path   = 'NSCLC_External_Dataset'
save_path   = './Nodule_crop_NSCLC'
csv_file_path = 'External_NSCLC_Radiomics.csv'

os.makedirs(save_path, exist_ok=True)
with open(csv_file_path, 'w', newline='') as f:
    csv.writer(f).writerow(['Patient_ID', 'Nodule No', 'Image File', 'Mask File', 'Malignancy'])

all_series = glob.glob(os.path.join(home_path, '*'))
patient_to_ct = {}
patient_to_rt = {}
print("Scanning folders...")
for series_dir in all_series:
    if not os.path.isdir(series_dir): continue
    dcms = glob.glob(os.path.join(series_dir, '*.dcm'))
    if not dcms: continue
    try:
        dcm = dicom.dcmread(dcms[0], stop_before_pixels=True)
        pid, mod = dcm.PatientID, dcm.Modality
        if mod == 'CT':          patient_to_ct[pid] = series_dir
        elif mod == 'RTSTRUCT':  patient_to_rt[pid] = series_dir
    except: pass

tasks = [(pid, patient_to_ct[pid],
          glob.glob(os.path.join(patient_to_rt[pid], '*.dcm'))[0],
          save_path)
         for pid in patient_to_ct if pid in patient_to_rt]

n_cores = cpu_count()
print(f"Found {len(tasks)} patients. Using {n_cores} CPU cores in parallel...")

with Pool(processes=n_cores) as pool:
    for i, (pid, count, results) in enumerate(pool.imap_unordered(process_patient, tasks), 1):
        if count == -1:
            print(f"✗ [{i}/{len(tasks)}] Error on {pid}: {results}")
        elif count == 0:
            print(f"✗ [{i}/{len(tasks)}] No nodule centers: {pid}")
        else:
            with open(csv_file_path, 'a', newline='') as f:
                for row in results:
                    csv.writer(f).writerow(row)
            print(f"✓ [{i}/{len(tasks)}] {pid} — {count} nodule(s)")

print("\nDone! Check Nodule_crop_NSCLC/ and External_NSCLC_Radiomics.csv")


Scanning folders...
Found 417 patients. Using 48 CPU cores in parallel...
✓ [1/417] LUNG1-138 — 1 nodule(s)
✓ [2/417] LUNG1-080 — 1 nodule(s)
✓ [3/417] LUNG1-032 — 1 nodule(s)
✓ [4/417] LUNG1-308 — 1 nodule(s)
✓ [5/417] LUNG1-204 — 1 nodule(s)
✓ [6/417] LUNG1-183 — 1 nodule(s)
✓ [7/417] LUNG1-333 — 1 nodule(s)
✓ [8/417] LUNG1-076 — 1 nodule(s)
✓ [9/417] LUNG1-326 — 1 nodule(s)
✓ [10/417] LUNG1-157 — 1 nodule(s)
✓ [11/417] LUNG1-028 — 1 nodule(s)
✓ [12/417] LUNG1-033 — 1 nodule(s)
✓ [13/417] LUNG1-148 — 1 nodule(s)
✓ [14/417] LUNG1-321 — 1 nodule(s)
✓ [15/417] LUNG1-166 — 1 nodule(s)
✓ [16/417] LUNG1-407 — 1 nodule(s)
✓ [17/417] LUNG1-260 — 1 nodule(s)
✓ [18/417] LUNG1-022 — 1 nodule(s)
✓ [19/417] LUNG1-190 — 1 nodule(s)
✓ [20/417] LUNG1-036 — 1 nodule(s)
✓ [21/417] LUNG1-258 — 1 nodule(s)
✓ [22/417] LUNG1-179 — 1 nodule(s)
✓ [23/417] LUNG1-306 — 1 nodule(s)
✓ [24/417] LUNG1-207 — 1 nodule(s)
✓ [25/417] LUNG1-078 — 1 nodule(s)
✓ [26/417] LUNG1-257 — 1 nodule(s)
✓ [27/417] LUNG1-209 — 1 

In [1]:
!pip install git+https://github.com/AIM-Harvard/pyradiomics.git

  Cloning https://github.com/AIM-Harvard/pyradiomics.git to /tmp/pip-req-build-yjqr0xc6
  Running command git clone --filter=blob:none --quiet https://github.com/AIM-Harvard/pyradiomics.git /tmp/pip-req-build-yjqr0xc6
  Resolved https://github.com/AIM-Harvard/pyradiomics.git to commit 8ed579383b44806651c463d5e691f3b2b57522ab
  DEPRECATION: Setting PIP_CONSTRAINT will not affect build constraints in the future, pip 26.2 will enforce this behaviour change. A possible replacement is to specify build constraints using --build-constraint or PIP_BUILD_CONSTRAINT. To disable this warning without any build constraints set --use-feature=build-constraint or PIP_USE_FEATURE="build-constraint".
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... 

In [4]:
!pip install uv

In [2]:
!uv pip install --system --break-system-packages transformers==4.38.0 xgboost lightgbm scikit-learn

Using Python 3.12.3 environment at: /usr
Resolved 26 packages in 1.40s                                        
Prepared 8 packages in 27.79s                                            
Uninstalled 2 packages in 75ms
Installed 8 packages in 127ms                               
 - huggingface-hub==1.4.1
 + huggingface-hub==0.36.2
 + lightgbm==4.7.0
 + narwhals==2.26.0
 + nvidia-nccl-cu13==2.32.3
 + pyyaml==6.0.3
 - tokenizers==0.22.2
 + tokenizers==0.15.2
 + transformers==4.38.0
 + xgboost==3.4.1


In [4]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoModel
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. LOAD THE ENSEMBLE MODEL
print("Loading Multimodal Ensemble...")
model_data = joblib.load("models/Proposed_Multimodal_Ensemble.pkl")
vit_idx = model_data['vit_idx']
xgb_clf = model_data['xgb']
lgb_clf = model_data['lgb']
print(f"Loaded ensemble! Optimal ViT Fold used: {vit_idx}")

# 2. ALIGN RADIOMICS COLUMNS
print("Aligning PyRadiomics features...")
df_baseline = pd.read_csv("Radiomics_Features_Base.csv")
exclude_cols = ['Patient_id', 'nodule no', 'File_Path', 'Malignancy', 'clean_pid', 'visit_id']
rad_features_cols = [c for c in df_baseline.columns if c not in exclude_cols]

new_df = pd.read_csv("External_NSCLC_Radiomics_Features.csv")

# Ensure columns match exactly (fill missing with 0 for safety)
missing_cols = [c for c in rad_features_cols if c not in new_df.columns]
if missing_cols:
    for c in missing_cols: 
        new_df[c] = 0.0
    
X_rad_new = new_df[rad_features_cols].values.astype(float)

# 3. LOAD IMAGES FOR ViT
pad_transform = transforms.Compose([
    transforms.Pad((80, 80, 80, 80), fill=0),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

class ExternalViTDataset(Dataset):
    def __init__(self, df): 
        self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = row['Patient_ID']
        nodule_no = int(row['Nodule_No'])  # Cast to int to prevent string formatting issues
        
        # Load the 3D numpy array - Fixed Path format
        vol_path = f"./Nodule_crop_NSCLC/{pid}/{pid}_{nodule_no}_roi.npy"
        vol = np.load(vol_path).astype(np.float32)
        
        depths = [16, 24, 32, 40, 48]
        tensor_list = [pad_transform(torch.tensor(vol[d-1:d+2, :, :], dtype=torch.float32)) for d in depths]
        return torch.stack(tensor_list)

# 4. EXTRACT ViT FEATURES
class MedicalVisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("microsoft/rad-dino")
        self.classifier = nn.Identity()
    def forward(self, x):
        return self.backbone(pixel_values=x).last_hidden_state[:,0,:]

print("Loading ViT Extractor...")
extractor = MedicalVisionTransformer().to(device)
weight_path = f"models/vit_fold_{vit_idx}.pt"
extractor.load_state_dict(torch.load(weight_path, map_location=device), strict=False)
extractor.eval()

test_dl = DataLoader(ExternalViTDataset(new_df), batch_size=8, shuffle=False, num_workers=0)

print(f"Extracting ViT features for {len(new_df)} nodules...")
vit_features_list = []
with torch.no_grad():
    for inputs in test_dl:
        batch_size = inputs.size(0)
        flat_inputs = inputs.view(-1, 3, 224, 224).to(device)
        flat_feats = extractor(flat_inputs)
        pooled_feats = flat_feats.view(batch_size, 5, 768).mean(dim=1)
        vit_features_list.append(pooled_feats.cpu().numpy())

X_vit_new = np.vstack(vit_features_list)

# 5. FINAL INFERENCE
print("Running Ensemble Prediction...")
X_fused_new = np.hstack([X_vit_new, X_rad_new])
y_true_new = new_df['Malignancy'].values.astype(int)  # All 1s

p_xgb = xgb_clf.predict_proba(X_fused_new)[:, 1]
p_lgb = lgb_clf.predict_proba(X_fused_new)[:, 1]
p_ensemble = (p_xgb + p_lgb) / 2.0

# Calculate Metrics
# Threshold of 0.5 for classification
preds = (p_ensemble >= 0.5).astype(int)
acc = accuracy_score(y_true_new, preds)
sens = sum((preds == 1) & (y_true_new == 1)) / sum(y_true_new == 1)

print("\n=========================================================")
print("FINAL EXTERNAL VALIDATION RESULTS (NSCLC-Radiomics)")
print("=========================================================")
print(f"Total Nodules Tested : {len(new_df)}")
print(f"Accuracy             : {acc:.4f}")
print(f"Sensitivity (Recall) : {sens:.4f}")
print("=========================================================")

Loading Multimodal Ensemble...
Loaded ensemble! Optimal ViT Fold used: 4
Aligning PyRadiomics features...
Loading ViT Extractor...
Extracting ViT features for 624 nodules...
Running Ensemble Prediction...

FINAL EXTERNAL VALIDATION RESULTS (NSCLC-Radiomics)
Total Nodules Tested : 624
Accuracy             : 0.0000
Sensitivity (Recall) : 0.0000


In [5]:
import pandas as pd

# Load both CSVs
df_baseline = pd.read_csv("Radiomics_Features_Base.csv")
new_df = pd.read_csv("External_NSCLC_Radiomics_Features.csv")

exclude_cols = ['Patient_id', 'nodule no', 'File_Path', 'Malignancy', 'clean_pid', 'visit_id']
rad_features_cols = [c for c in df_baseline.columns if c not in exclude_cols]
missing_cols = [c for c in rad_features_cols if c not in new_df.columns]

print(f"Base Radiomics Features Required: {len(rad_features_cols)}")
print(f"Features completely MISSING in new data: {len(missing_cols)}")
if len(missing_cols) > 0:
    print(f"\nExample missing features: {missing_cols[:5]}")
    
print(f"\nAverage Predicted Probability: {p_ensemble.mean():.4f}")
print(f"Min Probability: {p_ensemble.min():.4f} | Max Probability: {p_ensemble.max():.4f}")

Base Radiomics Features Required: 1218
Features completely MISSING in new data: 1118

Example missing features: ['log-sigma-1-0-mm-3D_firstorder_10Percentile', 'log-sigma-1-0-mm-3D_firstorder_90Percentile', 'log-sigma-1-0-mm-3D_firstorder_Energy', 'log-sigma-1-0-mm-3D_firstorder_Entropy', 'log-sigma-1-0-mm-3D_firstorder_InterquartileRange']

Average Predicted Probability: 0.0071
Min Probability: 0.0008 | Max Probability: 0.0875


In [8]:
!uv pip install --system --break-system-packages git+https://github.com/AIM-Harvard/pyradiomics.git SimpleITK

Using Python 3.12.3 environment at: /usr
Resolved 9 packages in 13.24s                                        
Prepared 1 package in 4.40s                                              
Installed 6 packages in 8ms                                 
 + docopt==0.6.2
 + pykwalify==1.8.0
 + pyradiomics==3.1.1.dev111+g8ed579383 (from git+https://github.com/AIM-Harvard/pyradiomics.git@8ed579383b44806651c463d5e691f3b2b57522ab)
 + pywavelets==1.10.0
 + ruamel-yaml==0.19.1
 + simpleitk==2.5.6


In [9]:
import numpy as np
import pandas as pd
import os
import glob
from radiomics import featureextractor
import SimpleITK as sitk
import warnings
warnings.filterwarnings('ignore')

# 1. Write your exact original configuration to the server
yaml_content = """
imageType:
  Original: {}
  LoG:
    sigma: [1.0, 2.0, 3.0, 4.0, 5.0]
  Wavelet: {}

featureClass:
  firstorder: []
  shape: []
  glcm: []
  glrlm: []
  glszm: []
  ngtdm: []
  gldm: []

setting:
  binWidth: 25
  label: 1
  interpolator: 'sitkBSpline'
  resampledPixelSpacing: [1, 1, 1]
  normalize: false
  force2D: false
"""
with open("params.yaml", "w") as f:
    f.write(yaml_content)

# 2. Initialize extractor with your exact parameters
extractor = featureextractor.RadiomicsFeatureExtractor("params.yaml")

save_path = './Nodule_crop_NSCLC'
rows = []
patient_folders = sorted(glob.glob(os.path.join(save_path, '*')))

print(f"Extracting FULL 1,218 PyRadiomics features for {len(patient_folders)} patients...")

for ppath in patient_folders:
    pid = os.path.basename(ppath)
    roi_files = sorted(glob.glob(os.path.join(ppath, '*_roi.npy')))
    mask_files = sorted(glob.glob(os.path.join(ppath, '*_label.npy')))

    for roi_path, mask_path in zip(roi_files, mask_files):
        try:
            nodule_num = os.path.basename(roi_path).split('_')[1]

            roi  = np.load(roi_path).astype(np.float32)
            mask = np.load(mask_path).astype(np.uint8)
            mask = (mask > 0.5).astype(np.uint8)

            roi_sitk  = sitk.GetImageFromArray(roi)
            mask_sitk = sitk.GetImageFromArray(mask)
            
            # Ensure spacing is 1,1,1 to match your yaml configuration
            roi_sitk.SetSpacing((1.0, 1.0, 1.0))
            mask_sitk.SetSpacing((1.0, 1.0, 1.0))

            result = extractor.execute(roi_sitk, mask_sitk)

            row = {'Patient_ID': pid, 'Nodule_No': nodule_num, 'Malignancy': 1}
            for key, val in result.items():
                if not key.startswith('diagnostics'):
                    row[key] = float(val) if val is not None else np.nan
            rows.append(row)
            
        except Exception as e:
            pass
            
    print(f"✓ Processed {pid}")

df = pd.DataFrame(rows)
df.to_csv('External_NSCLC_Radiomics_Features.csv', index=False)
print(f"\nDone! {len(df)} nodules extracted with {len(df.columns)} columns → External_NSCLC_Radiomics_Features.csv")

Extracting FULL 1,218 PyRadiomics features for 417 patients...


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

✓ Processed LUNG1-001


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

✓ Processed LUNG1-002


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

... (671 outputs left out)

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

✓ Processed LUNG1-421


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

✓ Processed LUNG1-422

Done! 624 nodules extracted with 1319 columns → External_NSCLC_Radiomics_Features.csv


In [12]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoModel
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def normalize_hu(image):
    MIN_BOUND = -1200.0
    MAX_BOUND = 600.0
    img = (image - MIN_BOUND) / (MAX_BOUND - MIN_BOUND)
    img[img > 1] = 1.0
    img[img < 0] = 0.0
    return img

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 1. LOAD THE ENSEMBLE MODEL
print("Loading Multimodal Ensemble...")
model_data = joblib.load("models/Proposed_Multimodal_Ensemble.pkl")
vit_idx = model_data['vit_idx']
xgb_clf = model_data['xgb']
lgb_clf = model_data['lgb']
print(f"Loaded ensemble! Optimal ViT Fold used: {vit_idx}")

# 2. ALIGN RADIOMICS COLUMNS
print("Aligning PyRadiomics features...")
df_baseline = pd.read_csv("Radiomics_Features_Base.csv")
exclude_cols = ['Patient_id', 'nodule no', 'File_Path', 'Malignancy', 'clean_pid', 'visit_id']
rad_features_cols = [c for c in df_baseline.columns if c not in exclude_cols]

new_df = pd.read_csv("External_NSCLC_Radiomics_Features.csv")

# Ensure columns match exactly
missing_cols = [c for c in rad_features_cols if c not in new_df.columns]
if missing_cols:
    for c in missing_cols: 
        new_df[c] = 0.0
    
X_rad_new = new_df[rad_features_cols].values.astype(float)

# 3. LOAD IMAGES FOR ViT
pad_transform = transforms.Compose([
    transforms.Pad((80, 80, 80, 80), fill=0),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

class ExternalViTDataset(Dataset):
    def __init__(self, df): 
        self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = row['Patient_ID']
        nodule_no = int(row['Nodule_No']) 
        
        # Load the 3D numpy array
        vol_path = f"./Nodule_crop_NSCLC/{pid}/{pid}_{nodule_no}_roi.npy"
        vol = np.load(vol_path).astype(np.float32)
        vol = normalize_hu(vol)  # THIS IS THE CRITICAL NORMALIZATION FIX
        
        depths = [16, 24, 32, 40, 48]
        tensor_list = [pad_transform(torch.tensor(vol[d-1:d+2, :, :], dtype=torch.float32)) for d in depths]
        return torch.stack(tensor_list)

# 4. EXTRACT ViT FEATURES
class MedicalVisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("microsoft/rad-dino")
        self.classifier = nn.Identity()
    def forward(self, x):
        return self.backbone(pixel_values=x).last_hidden_state[:,0,:]

print("Loading ViT Extractor...")
extractor = MedicalVisionTransformer().to(device)
weight_path = f"models/vit_fold_{vit_idx}.pt"
extractor.load_state_dict(torch.load(weight_path, map_location=device), strict=False)
extractor.eval()

test_dl = DataLoader(ExternalViTDataset(new_df), batch_size=8, shuffle=False, num_workers=0)

print(f"Extracting ViT features for {len(new_df)} nodules...")
vit_features_list = []
with torch.no_grad():
    # Added tqdm progress bar here!
    for inputs in tqdm(test_dl, desc="ViT Inference"):
        batch_size = inputs.size(0)
        flat_inputs = inputs.view(-1, 3, 224, 224).to(device)
        flat_feats = extractor(flat_inputs)
        pooled_feats = flat_feats.view(batch_size, 5, 768).mean(dim=1)
        vit_features_list.append(pooled_feats.cpu().numpy())

X_vit_new = np.vstack(vit_features_list)

# 5. FINAL INFERENCE
print("\nRunning Ensemble Prediction...")
X_fused_new = np.hstack([X_vit_new, X_rad_new])
y_true_new = new_df['Malignancy'].values.astype(int)  # All 1s

p_xgb = xgb_clf.predict_proba(X_fused_new)[:, 1]
p_lgb = lgb_clf.predict_proba(X_fused_new)[:, 1]
p_ensemble = (p_xgb + p_lgb) / 2.0

# Calculate Metrics
preds = (p_ensemble >= 0.5).astype(int)
acc = accuracy_score(y_true_new, preds)
sens = sum((preds == 1) & (y_true_new == 1)) / sum(y_true_new == 1)

print("\n=========================================================")
print("FINAL EXTERNAL VALIDATION RESULTS (NSCLC-Radiomics)")
print("=========================================================")
print(f"Total Nodules Tested : {len(new_df)}")
print(f"Accuracy             : {acc:.4f}")
print(f"Sensitivity (Recall) : {sens:.4f}")
print("=========================================================")

Using device: cuda
Loading Multimodal Ensemble...
Loaded ensemble! Optimal ViT Fold used: 4
Aligning PyRadiomics features...
Loading ViT Extractor...
Extracting ViT features for 624 nodules...


ViT Inference: 100%|████████████████████████████████████████████| 78/78 [00:04<00:00, 16.82it/s]


Running Ensemble Prediction...

FINAL EXTERNAL VALIDATION RESULTS (NSCLC-Radiomics)
Total Nodules Tested : 624
Accuracy             : 0.0000
Sensitivity (Recall) : 0.0000


In [13]:
import pandas as pd
import numpy as np

# 1. Check Columns
df_baseline = pd.read_csv("Radiomics_Features_Base.csv")
new_df = pd.read_csv("External_NSCLC_Radiomics_Features.csv")

exclude_cols = ['Patient_id', 'nodule no', 'File_Path', 'Malignancy', 'clean_pid', 'visit_id']
rad_features_cols = [c for c in df_baseline.columns if c not in exclude_cols]
missing_cols = [c for c in rad_features_cols if c not in new_df.columns]

print(f"Number of missing columns: {len(missing_cols)}")
if len(missing_cols) > 0:
    print(f"Example missing: {missing_cols[:5]}")
else:
    print("ALL COLUMNS MATCH PERFECTLY!")

# 2. Check Data
X_rad_new = new_df[rad_features_cols].values.astype(float)
print(f"\nRadiomics Matrix shape: {X_rad_new.shape}")
print(f"Percentage of matrix that is exactly zero: {(X_rad_new == 0).mean() * 100:.2f}%")

# 3. Check Probabilities
print(f"\nMean XGBoost Probability: {p_xgb.mean():.4f}")
print(f"Mean LightGBM Probability: {p_lgb.mean():.4f}")
print(f"Mean Ensemble Probability: {p_ensemble.mean():.4f}")
print(f"Max Ensemble Probability: {p_ensemble.max():.4f}")

Number of missing columns: 0
ALL COLUMNS MATCH PERFECTLY!

Radiomics Matrix shape: (624, 1218)
Percentage of matrix that is exactly zero: 1.11%

Mean XGBoost Probability: 0.0527
Mean LightGBM Probability: 0.0044
Mean Ensemble Probability: 0.0286
Max Ensemble Probability: 0.1476


In [14]:
print("Sensitivity (Accuracy) at different Decision Thresholds:")
print("------------------------------------------------------")
for t in [0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.12, 0.50]:
    preds_t = (p_ensemble >= t).astype(int)
    sens_t = sum((preds_t == 1) & (y_true_new == 1)) / sum(y_true_new == 1)
    print(f"Threshold >= {t:.2f}  --->  Accuracy: {sens_t*100:.2f}%")

Sensitivity (Accuracy) at different Decision Thresholds:
------------------------------------------------------
Threshold >= 0.01  --->  Accuracy: 93.75%
Threshold >= 0.02  --->  Accuracy: 56.89%
Threshold >= 0.03  --->  Accuracy: 33.01%
Threshold >= 0.05  --->  Accuracy: 10.90%
Threshold >= 0.08  --->  Accuracy: 3.04%
Threshold >= 0.10  --->  Accuracy: 1.60%
Threshold >= 0.12  --->  Accuracy: 0.96%
Threshold >= 0.50  --->  Accuracy: 0.00%
